Pilot Analysis 

In [ ]:

import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt

gt = pd.read_csv(r'/mnt/data/pilot_ground_truth_raw.csv')
llm = pd.read_csv(r'/mnt/data/pilot_llm_output_raw.csv')

gt['issue_key'] = (
    gt['BUG-ID'].astype(str)
    .str.replace(' Raw','', regex=False)
    .str.replace(' Improved','', regex=False)
)

df = gt.merge(llm, on='issue_key', how='inner')

print("Ground Truth:", len(gt))
print("LLM Output:", len(llm))
print("Merged:", len(df))

df.head()


In [ ]:

LABEL_MAP = {
    'S2R Label':'s2r_label',
    'S2R Irrep Category':'s2r_failure_type',
    'OB Category':'observed_behavior_presence',
    'OB Label':'observed_behavior_quality',
    'EB Category':'expected_behavior_presence',
    'EB Label':'expected_behavior_quality'
}

results=[]

for gt_col,llm_col in LABEL_MAP.items():
    y_true=df[gt_col].fillna('MISSING').astype(str)
    y_pred=df[llm_col].fillna('MISSING').astype(str)

    results.append({
        'Dimension':gt_col,
        'Accuracy':accuracy_score(y_true,y_pred),
        'Precision(Macro)':precision_score(y_true,y_pred,average='macro',zero_division=0),
        'Recall(Macro)':recall_score(y_true,y_pred,average='macro',zero_division=0),
        'F1(Macro)':f1_score(y_true,y_pred,average='macro',zero_division=0)
    })

metrics_df=pd.DataFrame(results)
metrics_df


In [ ]:

# Average metrics across all annotation dimensions
metrics_df[['Accuracy','Precision(Macro)','Recall(Macro)','F1(Macro)']].mean().to_frame('Mean Score')


In [ ]:

# Quality scoring logic adapted from kappa_pilot.py

def score_row(s2r, irr, ob, obl, eb, ebl):
    s2r=str(s2r).strip().lower()
    irr=str(irr).strip().lower()
    ob=str(ob).strip().lower()
    obl=str(obl).strip().lower()
    eb=str(eb).strip().lower()
    ebl=str(ebl).strip().lower()

    if s2r == 'executable':
        if ob == 'present' and obl == 'sufficient' and eb == 'present' and ebl == 'accurate':
            return 5
        return 4

    if irr == 'wrong information':
        return 1

    if ob == 'present' and eb == 'present':
        return 3

    if ob == 'present' or eb == 'present':
        return 2

    return 1

df['gt_score'] = df.apply(
    lambda r: score_row(
        r['S2R Label'],
        r['S2R Irrep Category'],
        r['OB Category'],
        r['OB Label'],
        r['EB Category'],
        r['EB Label']
    ), axis=1
)

df['llm_score'] = df.apply(
    lambda r: score_row(
        r['s2r_label'],
        r['s2r_failure_type'],
        r['observed_behavior_presence'],
        r['observed_behavior_quality'],
        r['expected_behavior_presence'],
        r['expected_behavior_quality']
    ), axis=1
)

df[['issue_key','gt_score','llm_score']].head()


In [ ]:

from sklearn.metrics import accuracy_score

score_accuracy = accuracy_score(df['gt_score'], df['llm_score'])
score_f1 = f1_score(df['gt_score'], df['llm_score'], average='macro')

print("Quality Score Accuracy =", round(score_accuracy,4))
print("Quality Score F1 =", round(score_f1,4))

pd.crosstab(df['gt_score'], df['llm_score'],
            rownames=['Ground Truth'],
            colnames=['LLM'])


In [ ]:

# Histogram distribution required by Section 7.3

plt.figure(figsize=(8,5))
plt.hist(df['gt_score'], bins=np.arange(1,7)-0.5)
plt.title('Ground Truth Quality Score Distribution')
plt.xlabel('Score')
plt.ylabel('Frequency')
plt.show()

plt.figure(figsize=(8,5))
plt.hist(df['llm_score'], bins=np.arange(1,7)-0.5)
plt.title('LLM Quality Score Distribution')
plt.xlabel('Score')
plt.ylabel('Frequency')
plt.show()


In [ ]:

# Descriptive statistics used to verify statistical assumptions

display(df['gt_score'].describe())
display(df['llm_score'].describe())

print('Interpretation:')
print('- If distribution approximately follows expected assumptions, keep test from proposal.')
print('- Otherwise document deviation in notes.md, project log, and amendment if required.')


In [ ]:

# Cohen's Kappa (direct calculation)

from sklearn.metrics import cohen_kappa_score

kappa = cohen_kappa_score(df['gt_score'], df['llm_score'])
print("Cohen Kappa =", round(kappa,4))


In [ ]:

# Run official script

import subprocess, os

script='Scripts/kappa_pilot.py'

if os.path.exists(script):
    subprocess.run(['python', script])
else:
    print('Copy kappa_pilot.py into Scripts/ and run this cell.')
